## Check that the Window Dataset class works

In [1]:
import os
import pandas as pd
from tqdm import tqdm
if os.getcwd().endswith("notebooks"):
    os.chdir("../")

from src.embeddings.dataset import WindowDataset, WindowDatasetTCN
from src.embeddings.train import train_autoencoder
from torch.utils.data import DataLoader
from src.embeddings.models import MotionAutoencoder, MotionAutoencoderOld

In [2]:
processed_path = "data/processed"
paths = [os.path.join(processed_path, f) for f in os.listdir(processed_path) if f.endswith("10fps_processed.pkl")]

df_dict = {}
cols_to_keep = ['frame', 'hand_label', 'cx_smooth', 'cy_smooth']

# load df with where first column in csv serves as index
df_vid_name_map = pd.read_csv("data/scores/vid_name_map.csv", index_col=0)

with tqdm(total=len(paths), desc="Loading processed data") as pbar:
    for path in paths:
        vid = os.path.basename(path).replace("_10fps_processed.pkl", "")
        vid = vid.replace("hand_tracking_", "")
        participant_id = df_vid_name_map.loc[vid]['Participant Number']
        #if int(participant_id) == 8:
        #    continue
        df_dict[(vid, int(participant_id))] = pd.read_pickle(path)[cols_to_keep]
        pbar.update(1)

Loading processed data: 100%|██████████| 86/86 [00:11<00:00,  7.37it/s]


In [11]:
validation_goups = [1, 11, 17, 18]
train_groups = [g for g in range(1, 30) if (g not in validation_goups or g==8)]

train_df_dict = {key: df_dict[key] for key in df_dict.keys() if key[1] in train_groups}
val_df_dict = {key: df_dict[key] for key in df_dict.keys() if key[1] in validation_goups}

train_dataset = WindowDatasetTCN(
    df_dict=train_df_dict,
    window_size=20,
    step_size=10,
    hand="Right", 
    fps=10,
)

val_dataset = WindowDatasetTCN(
    df_dict=val_df_dict,
    window_size=20,
    step_size=10,
    hand="Right", 
    fps=10,
    scaling_stats=train_dataset.scaling_stats
)

Processing Right Hand Windows: 100%|██████████| 74/74 [00:00<00:00, 126.64it/s]


Computing robust scaling statistics from training data...


Processing Right Hand Windows: 100%|██████████| 12/12 [00:00<00:00, 110.96it/s]


In [4]:
import numpy as np

all_data = np.array([feats for feats, _ in train_dataset] + [feats for feats, _ in val_dataset])

# (N, T, C) -> (N x T, C)
all_data_reshaped = all_data.reshape(-1, all_data.shape[-1])

# turn into dataframe and describe
df_all_data = pd.DataFrame(all_data_reshaped, columns=["dx", "dy", "vx", "vy", "ax", "ay", "mask"])
df_all_data[df_all_data['mask']==1].describe()

,dx,dy,vx,vy,ax,ay,mask
count,690299.000000,690299.000000,690299.000000,690299.000000,690299.000000,690299.000000,690299.0
mean,0.057956,-0.031232,0.057956,-0.031232,-0.031529,-0.004615,1.0
std,1.628733,1.504155,1.628734,1.504156,1.582865,1.488726,0.0
min,-5.000000,-5.000000,-5.000000,-5.000000,-5.000000,-5.000000,1.0
25%,-0.414764,-0.469515,-0.414764,-0.469515,-0.488387,-0.495997,1.0
50%,-0.005668,0.000000,-0.005668,0.000000,0.007734,-0.007994,1.0
75%,0.517047,0.469508,0.517047,0.469508,0.457337,0.456015,1.0
max,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,1.0


Processing videos: 100%|██████████| 86/86 [00:02<00:00, 29.20it/s]


In [23]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

In [ ]:
ae = MotionAutoencoder(feature_dim=7, latent_dim=16, dropout_prob=0.1).to("cpu")
best_model, _ = train_autoencoder(ae, train_loader, val_loader=val_loader, epochs=50, lr=0.01)

Epoch 001: 100%|██████████| 952/952 [00:04<00:00, 193.03it/s]


Epoch 001 — train: 12.764852 | val: 242.586416


Epoch 002: 100%|██████████| 952/952 [00:05<00:00, 185.92it/s]


Epoch 002 — train: 11.830241 | val: 229.240870


Epoch 003: 100%|██████████| 952/952 [00:04<00:00, 205.62it/s]


...

Epoch 004 — train: 11.240224 | val: 220.712968


Epoch 005: 100%|██████████| 952/952 [00:04<00:00, 195.80it/s]


Epoch 005 — train: 10.908575 | val: 215.529580


Epoch 006: 100%|██████████| 952/952 [00:05<00:00, 174.62it/s]


KeyboardInterrupt: 

In [ ]:
ae = MotionAutoencoderOld(
        feature_dim=7,
        latent_dim=16,
        dropout_prob=0.1
    ).to("cpu")
best_model, _ = train_ae(train_dataset, feature_dim=7, latent_dim=16, batch_size=32, epochs=50, lr=1e-3, device="cpu")

KeyboardInterrupt: 

# Training of the Autoencoder

In [14]:
def train_ae(
    dataset,
    feature_dim=5,              # now includes "valid" channel
    latent_dim=16,
    batch_size=16,
    epochs=50,
    lr=1e-3,
    device="cpu",
    patience=10,
    delta=1e-5,
    lambda_aug=0.0,             # set > 0 to enable augmentation consistency loss
):
    """
    Train MotionAutoencoder with masked loss and optional synthetic gap augmentation.
    Assumes last feature in input is 'valid' mask.
    """
    
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    ae = MotionAutoencoder(
        feature_dim=feature_dim,
        latent_dim=latent_dim,
        dropout_prob=0.1 if lambda_aug > 0 else 0.0
    ).to(device)

    optimizer = torch.optim.Adam(ae.parameters(), lr=lr)

    best_loss = float("inf")
    best_state = None
    wait = 0

    ae.train()
    for epoch in range(epochs):

        epoch_loss = 0.0

        for x, _ in dataloader:

            x = x.to(device)

            motion = x[..., :-1]          # real motion
            valid = x[..., -1:]           # (batch, T, 1)

            optimizer.zero_grad()

            # ------ PASS 1: normal input ------
            out, latent_clean = ae(x)     # recon motion only

            # masked MSE
            diff = (out - motion) * valid
            mse_recon = (diff ** 2).sum() / (valid.sum() + 1e-6)

            # ------ PASS 2: augmented input ------
            if lambda_aug > 0:
                x_aug = x.clone()
                B, T, D = x_aug.shape

                # randomly drop some valid frames (fake tracking loss)
                drop_mask = (torch.rand(B, T, 1, device=device) < 0.1)
                x_aug[..., :-1] = x_aug[..., :-1] * (~drop_mask)   # zero out motion at dropped frames
                # keep valid mask unchanged

                _, latent_aug = ae(x_aug)

                # consistency loss
                mse_aug = torch.mean((latent_clean - latent_aug)**2)
            else:
                mse_aug = 0.0

            total_loss = mse_recon + lambda_aug * mse_aug
            total_loss.backward()
            optimizer.step()

            epoch_loss += total_loss.item() * x.size(0)

        epoch_loss /= len(dataset)
        print(f"Epoch {epoch+1:03d} — loss: {epoch_loss:.6f}")

    if best_state is not None:
        ae.load_state_dict(best_state)

    return ae, best_loss

In [12]:
dataset = WindowDataset(df_dict, hand="Right", feature_mode="pos_vel", window_size=20, step_size=5, orig_fps=30.0, max_gap_seconds=0.11, normalize=True)

train_dataset = WindowDataset(
    df_dict=train_df_dict,
    window_size=20,
    step_size=5,
    hand="Right", 
    orig_fps=30.0, max_gap_seconds=0.11, normalize=True
)

val_dataset = WindowDataset(
    df_dict=val_df_dict,
    window_size=20,
    step_size=5,
    hand="Right", 
    orig_fps=30.0, max_gap_seconds=0.11, normalize=True
)

Processing videos: 100%|██████████| 12/12 [00:00<00:00, 30.25it/s]


In [16]:
import numpy as np

all_data = np.array([feats for feats, _ in train_dataset] + [feats for feats, _ in val_dataset])

# (N, T, C) -> (N x T, C)
all_data_reshaped = all_data.reshape(-1, all_data.shape[-1])

# turn into dataframe and describe
df_all_data = pd.DataFrame(all_data_reshaped, columns=["x", "y", "vx", "vy", "mask"])
df_all_data[df_all_data['mask']==1].describe()

,x,y,vx,vy,mask
count,1.491516e+06,1.491516e+06,1.491516e+06,1.491516e+06,1491516.0
mean,3.923284e-01,3.497153e-01,3.320170e-03,-1.630157e-03,1.0
std,8.891776e-02,1.509243e-01,8.142461e-02,1.032245e-01,0.0
min,-4.970238e-03,-4.753968e-02,-1.846912e+00,-2.990741e+00,1.0
25%,3.404762e-01,2.119312e-01,-9.747148e-03,-1.970917e-02,1.0
50%,3.959375e-01,3.395767e-01,8.185208e-04,7.936358e-04,1.0
75%,4.514286e-01,4.622751e-01,1.458332e-02,2.037019e-02,1.0
max,9.231771e-01,9.634127e-01,2.122396e+00,2.200992e+00,1.0


In [13]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

In [15]:
import torch

#ae, _ = train_ae(dataset, feature_dim=5, latent_dim=16,
#             batch_size=32, epochs=80, lr=1e-3, device="cpu")

ae = MotionAutoencoderOld(
        feature_dim=5,
        latent_dim=16,
        dropout_prob=0.1
    ).to("cpu")

train_autoencoder(ae, train_loader, val_loader=val_loader, epochs=50, lr=0.01)

Epoch 001: 100%|██████████| 2082/2082 [00:11<00:00, 183.78it/s]


Epoch 001 — train: 0.019469 | val: 0.401140


Epoch 002: 100%|██████████| 2082/2082 [00:10<00:00, 192.93it/s]


Epoch 002 — train: 0.048083 | val: 0.960051


Epoch 003: 100%|██████████| 2082/2082 [00:10<00:00, 192.90it/s]


Epoch 003 — train: 0.048143 | val: 0.960808


Epoch 004: 100%|██████████| 2082/2082 [00:10<00:00, 198.29it/s]


Epoch 004 — train: 0.048131 | val: 0.966546


Epoch 005:  73%|███████▎  | 1516/2082 [00:08<00:03, 187.04it/s]


KeyboardInterrupt: 

In [2]:
# save the trained autoencoder model
import torch
model_path = "results/models/motion_autoencoder2.0.pth"
torch.save(ae.state_dict(), model_path)
print(f"Model saved to {model_path}")

NameError: name 'ae' is not defined

# Other TCN autoencoder with modified dataset function

In [2]:
from src.embeddings.dataset import WindowDatasetTCN
from src.embeddings.train import train_tcn_ae

## First Check That the Dataset class works

In [3]:
processed_path = "data/processed"
paths = [os.path.join(processed_path, f) for f in os.listdir(processed_path) if f.endswith("30fps_processed.pkl")]

df_dict = {}
cols_to_keep = ['frame', 'hand_label', 'cx_smooth', 'cy_smooth']

with tqdm(total=len(paths), desc="Loading processed data") as pbar:
    for path in paths:
        vid = os.path.basename(path).replace("_30fps_processed.pkl", "")
        vid = vid.replace("hand_tracking_", "")
        df_dict[vid] = pd.read_pickle(path)[cols_to_keep]
        pbar.update(1)

Loading processed data: 100%|██████████| 86/86 [00:34<00:00,  2.51it/s]


In [4]:
dataset = WindowDatasetTCN(df_dict, hand="Right", feature_mode="pos_vel_acc", window_size=90, step_size=30, orig_fps=30.0, scaling=True)

Processing Right Hand Windows: 100%|██████████| 86/86 [00:00<00:00, 96.06it/s] 


Computing scaling statistics from training data...


In [8]:
dataset[0][0]

tensor([[-0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00,  0.0000e+00,
         -0.0000e+00,  0.0000e+00],
        [ 0.0000e+00, -0.0000e+00,  0.0000e+00, -0.0000e+00,  0.0000e+00,
         -0.0000e+00,  0.0000e+00],
        [ 1.5087e+00, -4.6047e+00,  1.5087e+00, -4.6049e+00,  3.9479e-01,
          4.3950e+00,  1.0000e+00],
        [ 1.3498e+00, -1.7434e+00,  1.3498e+00, -1.7435e+00, -5.4350e-01,
          9.3230e+00,  1.0000e+00],
        [ 1.0930e+00, -4.0059e+00,  1.0930e+00, -4.0061e+00, -8.9448e-01,
         -7.4283e+00,  1.0000e+00],
        [-0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00,  0.0000e+00,
         -0.0000e+00,  0.0000e+00],
        [-0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00,  0.0000e+00,
         -0.0000e+00,  0.0000e+00],
        [-0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00,  0.0000e+00,
         -0.0000e+00,  0.0000e+00],
        [-0.0000e+00, -0.0000e+00, -0.0000e+00, -0.0000e+00,  0.0000e+00,
         -0.0000e+00,  0.0000e+00],
        [ 

In [6]:
ae, _ = train_tcn_ae(dataset, feature_dim=7, latent_dim=16,
             batch_size=32, epochs=20, lr=1e-3, device="cpu")

Epoch 001: 100%|██████████| 1100/1100 [01:25<00:00, 12.88it/s]


Epoch 001 — loss: 0.601497


Epoch 002: 100%|██████████| 1100/1100 [01:31<00:00, 12.00it/s]


Epoch 002 — loss: 0.537886


Epoch 003: 100%|██████████| 1100/1100 [01:35<00:00, 11.52it/s]


...

Epoch 011 — loss: 0.410934


Epoch 012: 100%|██████████| 1100/1100 [01:31<00:00, 12.07it/s]


Epoch 012 — loss: 0.407395


Epoch 013:   2%|▏         | 20/1100 [00:01<01:31, 11.82it/s]


KeyboardInterrupt: 